In [4]:
import pandas as pd

data = pd.read_csv('./data/train.csv')

In [5]:
pd.get_dummies(data['HomePlanet'], prefix='HomePlanet')

,HomePlanet_Earth,HomePlanet_Europa,HomePlanet_Mars
0,0,1,0
1,1,0,0
2,0,1,0
3,0,1,0
4,1,0,0
...,...,...,...
8688,0,1,0
8689,1,0,0
8690,1,0,0
8691,0,1,0


In [6]:
data = pd.read_csv('./data/train.csv')
# Add back in cabin once I have more time
data = data.drop(['PassengerId', 'Cabin', 'Name'], axis=1)
data = data.dropna()
data['HomePlanet'] = data['HomePlanet'].astype('category')
data['CryoSleep'] = data['CryoSleep'].astype('bool')
data['Destination'] = data['Destination'].astype('category')
data['VIP'] = data['VIP'].astype('bool')

data


,HomePlanet,CryoSleep,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported
0,Europa,False,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,False
1,Earth,False,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,True
2,Europa,False,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,False
3,Europa,False,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,False
4,Earth,False,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,True
...,...,...,...,...,...,...,...,...,...,...,...
8688,Europa,False,55 Cancri e,41.0,True,0.0,6819.0,0.0,1643.0,74.0,False
8689,Earth,True,PSO J318.5-22,18.0,False,0.0,0.0,0.0,0.0,0.0,False
8690,Earth,False,TRAPPIST-1e,26.0,False,0.0,0.0,1872.0,1.0,0.0,True
8691,Europa,False,55 Cancri e,32.0,False,0.0,1049.0,0.0,353.0,3235.0,False


In [7]:
data.dtypes

HomePlanet      category
CryoSleep           bool
Destination     category
Age              float64
VIP                 bool
RoomService      float64
FoodCourt        float64
ShoppingMall     float64
Spa              float64
VRDeck           float64
Transported         bool
dtype: object

In [8]:
pd.get_dummies(data['HomePlanet'])

,Earth,Europa,Mars
0,0,1,0
1,1,0,0
2,0,1,0
3,0,1,0
4,1,0,0
...,...,...,...
8688,0,1,0
8689,1,0,0
8690,1,0,0
8691,0,1,0


In [9]:
import torch

class Dataset(torch.utils.data.Dataset):
  def __init__(self, df):
    self.home_planet = pd.get_dummies(df['HomePlanet'], prefix='HomePlanet')
    self.destination = pd.get_dummies(df['Destination'], prefix='Destination')

    self.vip = df['VIP']
    self.cryo_sleep = df['CryoSleep']

    self.age = df['Age']
    self.spending = df[['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']]

    self.y = df['Transported']
    
  def __len__(self):
    return len(self.vip)
    
  def __getitem__(self, idx):
    if isinstance(idx, torch.Tensor):
            idx = idx.tolist()

    # Only working on home planet right now ....
    # TODO fix this to give everything
    return [self.home_planet[idx].values, self.y[idx]] 
    
dataset = Dataset(data)

In [14]:
from torch import nn

class Model(nn.Module):
  def __init__(self):
    super().__init__()
    self.home_planet = nn.Sequential(
      nn.Linear(3, 3),
      nn.ReLU(),
      nn.Linear(3, 1)
    )

  def forward(self, x):
    x = self.home_planet(x)
    return x.squeeze()

ValueError: No axis named 4155 for object type DataFrame

In [16]:
from torch.utils.data import DataLoader, random_split
import numpy as np
import matplotlib.pyplot as plt

def train(dataset, epochs=1_000):
  loader = DataLoader(dataset, batch_size=100, shuffle=True)
  
  model = Model()
  criterion = nn.MSELoss()
  optimizer = torch.optim.Adam(model.parameters(), weight_decay=0.0001)

  # Sad days :(
  device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
  
  # Train the net
  loss_per_iter = []
  loss_per_batch = []
  for epoch in range(epochs):

    running_loss = 0.0
    for i, (inputs, labels) in enumerate(loader):
      inputs = inputs.to(device)
      labels = labels.to(device)

      # Zero the parameter gradients
      optimizer.zero_grad()

      # Forward + backward + optimize
      outputs = model(inputs.float())
      loss = criterion(outputs, labels.float())
      loss.backward()
      optimizer.step()

      # Save loss to plot
      running_loss += loss.item()
      loss_per_iter.append(loss.item())

      loss_per_batch.append(running_loss / (i + 1))
      running_loss = 0.0

  # Comparing training to test
  # dataiter = iter(testloader)
  #inputs, labels = dataiter.next()
  inputs = inputs.to(device)
  labels = labels.to(device)
  # outputs = net(inputs.float())
  print("Root mean squared error")
  print("Training:", np.sqrt(loss_per_batch[-1]))
  print("Test", np.sqrt(criterion(labels.float(), outputs).detach().cpu().numpy()))

  # Plot training loss curve
  plt.plot(np.arange(len(loss_per_iter)), loss_per_iter, "-", alpha=0.5, label="Loss per epoch")
  plt.plot(np.arange(len(loss_per_iter), step=4) + 3, loss_per_batch, ".-", label="Loss per mini-batch")
  plt.xlabel("Number of epochs")
  plt.ylabel("Loss")
  plt.legend()
  plt.show()

train(dataset)



KeyError: 4155